# 🔎 Name That LLM | EDA + Where the Fingerprint Lives

### 18 frontier models, 150 prompts, one paragraph each: what gives a model away, and how to validate it so the board agrees

**Task:** name the model that wrote each passage · **Metric:** balanced accuracy over 18 models · **Data:** 2,699 labelled passages (150 prompts × 18 models), ~4,000 hidden test passages on unseen prompts

Every number below is measured in this notebook, on prompt-grouped folds, and scored the way the hidden test is built: **a few passages per prompt, not all eighteen.**

### 🔑 Key findings

**01 · Typography sorts the labs before a single word is read.** Gemini writes **0.00 em-dashes per 100 words**; Claude and GLM write **0.9-1.6** and **never** a curly apostrophe; every GPT model uses curly apostrophes in about half its passages. A classifier on punctuation shapes alone (letters masked) reaches **0.239** balanced accuracy, 4.3× chance (0.056). Length alone: 0.111.

**02 · The lab is easier than the model, but neither is solved.** Character TF-IDF names the **lab correctly 67.5%** of the time but the exact model only **0.329** (balanced). Of its wrong answers, **52%** stay inside the true lab, about twice what a random wrong guess would. Recall ranges from **0.147** (glm-5.3-flash) to **0.667** (gemini-3.1-pro-preview).

**03 · The prompt shouts louder than the author.** A passage's nearest neighbour (word TF-IDF) is another answer to **the same prompt 99.4%** of the time and a passage by **the same model 0.15%** of the time, below the 5.6% a random pick would give. Mask the letters and keep only punctuation shape, and the same-model rate rises to **14.4%**.

**04 · The centring trap.** Subtracting each prompt's mean score lifts balanced accuracy from **0.329** to **0.390** when all 18 answers per prompt are present, and *drops* it from 0.332 to **0.267** when only 2 are. The break-even is between 3 and 6 answers per prompt. The hidden test has few passages per prompt, which is why prompt-group tricks win offline and lose on the board. Score your folds on 2 answers per prompt (section 08 has the harness).

### 📑 Roadmap
[00 Poster](#00) → [01 Setup](#01) → [02 Data](#02) → [03 Length](#03) → [04 Typography](#04) → [05 Lab rules](#05) → [06 Openings](#06) → [07 Topic vs author](#07) → [08 CV harness](#08) → [09 Three text views](#09) → [10 Confusions](#10) → [11 Centring trap](#11) → [12 What it means](#12)

**Central question:** which signals identify the *author* rather than the *prompt*, and which validation tells you the truth about unseen prompts?

## Credits

| Whose work | What I took from it |
|---|---|
| **alexchilton**, [Name That LLM 0.680: All Public + What Failed](https://www.kaggle.com/code/alexchilton/name-that-llm-0-680-all-public-what-failed) | The board evidence that prompt-group centring and Hungarian assignment lose on the hidden test, and the resubmission-noise table (up to 0.009 on identical code). Section 11 reproduces the mechanism offline. |
| **GoPain**, comment on [discussion 753177](https://www.kaggle.com/competitions/name-that-llm/discussion/753177) (quoted by alexchilton) | Scoring out-of-fold on 2 answers per prompt instead of 18. The harness in section 08 is built on that idea; the code is mine. |
| **Kaggle / Yao Yan, Martyna Plomecka, Ashley Oldacre** | The competition and its data. |

## Version history

| version | what changed |
|---|---|
| v1 | private test run |
| v2 | measured numbers in the text; poster; typography fingerprints, lab rules, openings, topic-vs-author neighbours, prompt-grouped CV harness with 2-per-prompt scoring, three TF-IDF views, confusions, the centring trap |

Upvotes tell me the series is worth continuing; comments tell me what to measure next.

<a id='01'></a>
## 01 · Setup and visual style

One palette (one colour per lab), one figure helper: a bold left-aligned title that states the conclusion and a muted note that states the caveat.

In [ ]:
import os, glob, re, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score as bacc, confusion_matrix
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore")

SEED = 42
C = {"ink": "#1B2A41", "muted": "#6B7A90", "pale": "#F7F9FC", "grid": "#DCE3EC", "red": "#D9534F", "mint": "#1FA98E"}
LAB = {"OpenAI": "#10A37F", "Google": "#4285F4", "Anthropic": "#D97757", "Qwen": "#7B4FD6", "ZAI": "#E0A33B", "Moonshot": "#1B2A41"}
sns.set_theme(style="white", font_scale=0.95)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": C["pale"], "axes.edgecolor": C["grid"],
                     "axes.labelcolor": C["ink"], "text.color": C["ink"], "xtick.color": C["muted"],
                     "ytick.color": C["muted"], "axes.titlesize": 11, "axes.titleweight": "semibold",
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

def finish(fig, title, note="", top=0.88):
    fig.suptitle(title, x=0.03, y=0.99, ha="left", fontsize=15, fontweight="bold", color=C["ink"])
    if note: fig.text(0.03, 0.925, note, ha="left", fontsize=9, color=C["muted"])
    fig.tight_layout(rect=[0.01, 0.01, 0.99, top]); plt.show()

def find(name):
    hits = [p for p in glob.glob("/kaggle/input/**/" + name, recursive=True)]
    assert hits, f"missing {name}"; return hits[0]

def lab_of(m):
    return ("OpenAI" if m.startswith("gpt") else "Google" if m.startswith("gemini") else "Anthropic" if m.startswith("claude")
            else "Qwen" if m.startswith("qwen") else "ZAI" if m.startswith("glm") else "Moonshot")

<a id='02'></a>
## 02 · The data and its shape

Every prompt was answered by every model, so the training set is a perfect 150 × 18 grid. The hidden test is not: its prompts are unseen and its class mix differs between the public and private boards, which is why the metric is balanced accuracy.

In [ ]:
train = pd.read_csv(find("train.csv")); test = pd.read_csv(find("test.csv"))
y, g = train["model"].values, train["prompt_id"].values
MODELS = sorted(train["model"].unique()); ORDER = sorted(MODELS, key=lambda m: (lab_of(m), m))
train["lab"] = train["model"].map(lab_of)
train["words"] = train["text"].str.split().str.len()
grid = train.groupby("prompt_id")["model"].nunique()
summary = pd.DataFrame({
    "rows": [len(train), len(test)], "prompts": [train.prompt_id.nunique(), np.nan],
    "models": [len(MODELS), np.nan], "missing text": [int(train.text.isna().sum()), int(test.text.isna().sum())],
    "duplicate texts": [int(train.text.duplicated().sum()), int(test.text.duplicated().sum())]}, index=["train", "test (interactive copy)"])
display(summary); print(summary.to_string())
print("prompts with all 18 models:", int((grid == 18).sum()), "| with 17:", int((grid == 17).sum()))
print("rows per model:", train.model.value_counts().agg(["min", "max"]).to_dict())
print("words per passage: min %d, median %d, max %d" % (train.words.min(), train.words.median(), train.words.max()))

<a id='00'></a>
## 00 · The poster: author habits versus topic

Left: two punctuation habits, one dot per model. Right: what a passage's nearest neighbour shares with it, by words and by punctuation shape alone.

In [ ]:
fl = pd.DataFrame({"em": train.text.str.contains("—"), "cq": train.text.str.contains("[’“”]"), "model": y}).groupby("model").mean()
def nn_match(X):
    i = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X).kneighbors(X, return_distance=False)[:, 1]
    return (g[i] == g).mean(), (y[i] == y).mean()
w_p, w_m = nn_match(TfidfVectorizer(sublinear_tf=True, min_df=2).fit_transform(train.text))
mask = train.text.map(lambda t: re.sub(r"[A-Za-z0-9]+", "w", t))
p_p, p_m = nn_match(TfidfVectorizer(analyzer="char", ngram_range=(1, 4), sublinear_tf=True, min_df=2).fit_transform(mask))
fig, (a, b) = plt.subplots(1, 2, figsize=(15, 6.2), gridspec_kw={"width_ratios": [1.25, 1]})
for m in ORDER:
    a.scatter(fl.loc[m, "em"], fl.loc[m, "cq"], s=150, color=LAB[lab_of(m)], edgecolor="white", zorder=3)
    a.annotate(m, (fl.loc[m, "em"], fl.loc[m, "cq"]), xytext=(6, 4), textcoords="offset points", fontsize=7.5)
for lab, col in LAB.items(): a.scatter([], [], color=col, s=70, label=lab)
a.legend(frameon=False, loc="upper right", fontsize=8); a.grid(color=C["grid"], lw=.6)
a.set_xlabel("share of passages with an em-dash"); a.set_ylabel("share with a curly quote"); a.set_title("Every lab sits in its own corner")
vals = pd.DataFrame({"same prompt": [w_p, p_p], "same model": [w_m, p_m]}, index=["word TF-IDF", "punctuation shape"])
vals.plot.barh(ax=b, color=[C["red"], C["mint"]], width=.7)
for c in b.containers: b.bar_label(c, fmt="%.3f", padding=3, fontsize=9)
b.set_xlim(0, 1.15); b.set_xlabel("share of nearest neighbours that match"); b.legend(frameon=False, loc="lower right")
b.set_title("Words find the prompt; punctuation finds the author")
finish(fig, "Name That LLM: the author is in the punctuation, the prompt is in the words",
       "2,699 passages, 18 models, 150 prompts. Nearest neighbour = cosine over TF-IDF, excluding the passage itself.")

<a id='03'></a>
## 03 · Length: different, but not decisive

The host randomised passage length between 40 and 250 words so that length would not be a shortcut. The medians still differ; the question is whether it is enough to classify on.

In [ ]:
med = train.groupby("model")["words"].median().reindex(ORDER)
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=train, x="model", y="words", order=ORDER, palette=[LAB[lab_of(m)] for m in ORDER], fliersize=2, ax=ax)
ax.set_xlabel(""); ax.set_ylabel("words per passage"); ax.tick_params(axis="x", rotation=60)
for lbl in ax.get_xticklabels(): lbl.set_ha("right")

# length-only classifier on prompt-grouped folds
FOLDS = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(train, y, g))
Xlen = np.c_[np.log(train.words), np.log(train.text.str.len()), train.text.str.count(r"[.!?]")]
plen = np.empty(len(train), dtype=object)
for tr, va in FOLDS:
    plen[va] = LogisticRegression(max_iter=2000).fit(Xlen[tr], y[tr]).predict(Xlen[va])
LEN_BACC = bacc(y, plen)
finish(fig, f"Length differs by model, but alone it scores only {LEN_BACC:.3f}",
       "Box = quartiles of words per passage, 150 passages per model. Chance is 1/18 = 0.056. Folds grouped by prompt.")
print("length-only balanced accuracy: %.4f" % LEN_BACC)

<a id='04'></a>
## 04 · Typography: the fingerprint you can see

Rates per 100 words of characters a writer chooses without thinking: dashes, quote styles, semicolons, colons, parentheses. The heatmap is z-scored per column so each mark is compared across models; the printed numbers are the raw rates.

In [ ]:
MARKS = {"em-dash —": "—", "en-dash –": "–", "curly ’": "’", "straight '": "'", "curly “ ”": "[“”]", 'straight "': '"',
         "semicolon ;": ";", "colon :": ":", "paren (": r"\(", "question ?": r"\?", "hyphenated w-w": r"\w-\w", "digit": r"\d"}
rates = pd.DataFrame({k: train.text.str.count(p) / train.words * 100 for k, p in MARKS.items()})
rates["model"] = train.model
R = rates.groupby("model").mean().reindex(ORDER)
fig, ax = plt.subplots(figsize=(13, 8))
Z = (R - R.mean()) / R.std()
sns.heatmap(Z, annot=R.round(2), fmt="", cmap="RdBu_r", center=0, cbar=False, linewidths=.5, linecolor="white", ax=ax,
            annot_kws={"fontsize": 8})
for t in ax.get_yticklabels(): t.set_color(LAB[lab_of(t.get_text())]); t.set_fontweight("bold")
ax.set_xlabel(""); ax.set_ylabel("")
finish(fig, "Gemini never uses an em-dash; Claude and GLM never use a curly apostrophe",
       "Numbers = mean marks per 100 words (150 passages per model). Colour = z-score within each column. Model names coloured by lab.")
print(R.round(2).to_string())

<a id='05'></a>
## 05 · Two yes/no questions that split the labs

Does the passage contain an em-dash? Does it contain a curly quote? The share of passages answering *yes*, per model. These are not rules a model must follow, they are habits; but they are strong ones.

In [ ]:
flags = pd.DataFrame({"has em-dash": train.text.str.contains("—"), "has curly quote": train.text.str.contains("[’“”]"),
                      "model": train.model})
F = flags.groupby("model").mean().reindex(ORDER)
fig, ax = plt.subplots(figsize=(10, 7.5))
for m in ORDER:
    ax.scatter(F.loc[m, "has em-dash"], F.loc[m, "has curly quote"], s=140, color=LAB[lab_of(m)], edgecolor="white", zorder=3)
    ax.annotate(m, (F.loc[m, "has em-dash"], F.loc[m, "has curly quote"]), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("share of passages with an em-dash"); ax.set_ylabel("share of passages with a curly quote")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 0.75); ax.grid(color=C["grid"], lw=.6)
for lab, col in LAB.items(): ax.scatter([], [], color=col, s=80, label=lab)
ax.legend(frameon=False, loc="upper right")
finish(fig, "Two punctuation habits place every lab in its own corner",
       "Each dot = one model, 150 passages. Habits, not rules: short passages may contain neither mark.")
lab_flags = flags.assign(lab=train.lab).groupby("lab")[["has em-dash", "has curly quote"]].mean()
print(lab_flags.round(3).to_string())

<a id='06'></a>
## 06 · How each model starts

The first two words of a passage, most common per model. Openings carry little topic and a lot of habit.

In [ ]:
first2 = train.text.str.strip().str.split().str[:2].str.join(" ").str.lower().str.replace(r"[^\w\s'’-]", "", regex=True)
open_tab = (pd.DataFrame({"model": train.model, "first2": first2}).groupby("model")["first2"]
            .agg(lambda s: " · ".join(f"{k} ({v})" for k, v in s.value_counts().head(3).items())).reindex(ORDER))
display(open_tab.to_frame("top 3 openings (count of 150)")); print(open_tab.to_string())

<a id='07'></a>
## 07 · Topic versus author: who is your nearest neighbour?

For each passage, find its nearest neighbour by word TF-IDF cosine among all other passages. Does the neighbour answer the same prompt, or come from the same model? If topic dominates, a word model will learn prompts, and prompts are exactly what the test changes.

In [ ]:
def nn_share(X):
    nn = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X)
    idx = nn.kneighbors(X, return_distance=False)[:, 1]
    return (g[idx] == g).mean(), (y[idx] == y).mean(), (train.lab.values[idx] == train.lab.values).mean()
Xw = TfidfVectorizer(sublinear_tf=True, min_df=2).fit_transform(train.text)
masked = train.text.map(lambda t: re.sub(r"[A-Za-z0-9]+", "w", t))
Xp = TfidfVectorizer(analyzer="char", ngram_range=(1, 4), sublinear_tf=True, min_df=2).fit_transform(masked)
nnt = pd.DataFrame([nn_share(Xw), nn_share(Xp)], columns=["same prompt", "same model", "same lab"],
                   index=["word TF-IDF", "punctuation shape (letters masked)"])
NNP, NNM = nnt.iloc[0, 0], nnt.iloc[0, 1]
fig, ax = plt.subplots(figsize=(11, 4.2))
nnt.T.plot.barh(ax=ax, color=[C["red"], C["mint"]], width=.7)
ax.axvline(17/2698, color=C["muted"], ls=":", lw=1); ax.axvline(1/18, color=C["muted"], ls="--", lw=1)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", padding=3, fontsize=9)
ax.set_xlim(0, 1.05); ax.set_xlabel("share of passages whose nearest neighbour matches"); ax.legend(frameon=False, loc="lower right")
finish(fig, f"Words find the prompt ({NNP:.0%}), not the author ({NNM:.0%}); punctuation finds the author",
       "Nearest neighbour by cosine distance over all 2,699 passages. Dotted: random same-prompt rate 0.006; dashed: random same-model rate 0.056.")
print(nnt.round(4).to_string())

<a id='08'></a>
## 08 · The CV harness: grouped folds, scored on 2 answers per prompt

* **Folds:** `StratifiedGroupKFold(5, shuffle=True, random_state=42)` grouped by `prompt_id`, so no prompt is ever in both train and validation. These fold ids are written to `folds.csv`; every notebook in this series uses them.
* **Score:** balanced accuracy over all out-of-fold rows, and **2-per-prompt** balanced accuracy: draw 2 answers per prompt, score, repeat 20 times, report mean ± std. For a per-passage model both agree; for anything that uses other passages of the same prompt (centring, assignment, clustering) only the second one is honest about the hidden test.

This cell is self-contained: fork it.

In [ ]:
fold_id = np.empty(len(train), dtype=int)
for k, (tr, va) in enumerate(FOLDS): fold_id[va] = k
pd.DataFrame({"id": train.id, "prompt_id": g, "model": y, "fold": fold_id}).to_csv("folds.csv", index=False)
assert all(len(set(g[tr]) & set(g[va])) == 0 for tr, va in FOLDS)

def score_2pp(pred, draws=20):
    s = [bacc(y[i], pred[i]) for i in (train.groupby("prompt_id").sample(2, random_state=r).index.values for r in range(draws))]
    return float(np.mean(s)), float(np.std(s))

def oof_scores(make_xy, C_=10.0):
    S = np.zeros((len(train), len(MODELS)))
    for tr, va in FOLDS:
        Xtr, Xva = make_xy(train.text.iloc[tr], train.text.iloc[va])
        S[va] = LogisticRegression(C=C_, max_iter=3000).fit(Xtr, y[tr]).decision_function(Xva)
    return S
print("fold sizes:", np.bincount(fold_id).tolist(), "| prompts per fold:", [len(set(g[va])) for _, va in FOLDS])

<a id='09'></a>
## 09 · Three text views on the same folds

Logistic regression (C = 10) on three TF-IDF views, each fitted inside the training fold only:
1. **character 1-5 grams**, the classic authorship baseline;
2. **word 1-2 grams**, which section 07 says will chase topics;
3. **punctuation shape**: every run of letters and digits replaced by `w`, then character 1-4 grams. No vocabulary at all.

In [ ]:
def view(kind):
    def make(a, b):
        if kind == "punct":
            f = lambda s: s.map(lambda t: re.sub(r"[A-Za-z0-9]+", "w", t)); a, b = f(a), f(b)
            v = TfidfVectorizer(analyzer="char", ngram_range=(1, 4), sublinear_tf=True, min_df=2)
        elif kind == "char": v = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), sublinear_tf=True, min_df=2, max_features=300000)
        else: v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
        return v.fit_transform(a), v.transform(b)
    return make
CL = np.array(MODELS); OOF, rows = {}, []
for kind in ["char", "word", "punct"]:
    t = time.time(); S = oof_scores(view(kind)); p = CL[S.argmax(1)]; OOF[kind] = S
    m2, s2 = score_2pp(p)
    rows.append({"view": kind, "balanced acc (all OOF)": bacc(y, p), "2-per-prompt mean": m2, "2-per-prompt std": s2,
                 "lab accuracy": np.mean([lab_of(a) == lab_of(b) for a, b in zip(y, p)]), "seconds": time.time() - t})
views = pd.DataFrame(rows).set_index("view"); display(views.round(4)); print(views.round(4).to_string())
CHAR, PUNCT, FAM = views.loc["char", "balanced acc (all OOF)"], views.loc["punct", "balanced acc (all OOF)"], views.loc["char", "lab accuracy"]
pd.DataFrame(OOF["char"], columns=MODELS).assign(id=train.id.values).to_csv("oof_char_lr.csv", index=False)

<a id='10'></a>
## 10 · Where the character model goes wrong

Row-normalised confusion matrix of the character view (recall on the diagonal), ordered by lab. Blocks along the diagonal are families.

In [ ]:
pc = CL[OOF["char"].argmax(1)]
cm = confusion_matrix(y, pc, labels=ORDER, normalize="true")
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm, xticklabels=ORDER, yticklabels=ORDER, cmap="Blues", annot=True, fmt=".2f", annot_kws={"fontsize": 7},
            cbar=False, linewidths=.3, linecolor="white", ax=ax)
for t in ax.get_yticklabels() + ax.get_xticklabels(): t.set_color(LAB[lab_of(t.get_text())])
edges = np.cumsum(pd.Series([lab_of(m) for m in ORDER]).value_counts(sort=False).reindex(sorted(LAB)).dropna().values)
for e in edges[:-1]: ax.axhline(e, color=C["ink"], lw=1.2); ax.axvline(e, color=C["ink"], lw=1.2)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
wrong = pc != y; INFAM = np.mean([lab_of(a) == lab_of(b) for a, b in zip(y[wrong], pc[wrong])])
finish(fig, f"{INFAM:.0%} of mistakes name a sibling from the same lab", top=0.93,
       note="Row-normalised confusion, character TF-IDF + logistic regression, prompt-grouped out-of-fold predictions.")
recall = pd.Series(np.diag(cm), index=ORDER).sort_values()
print("per-model recall:\n" + recall.round(3).to_string())
print("share of errors inside the true lab: %.3f" % INFAM)

<a id='11'></a>
## 11 · The centring trap

A popular idea: answers to the same prompt share topic, so subtract each prompt's mean classifier score and what is left is style. Offline, with all 18 answers of every prompt in the validation fold, it looks brilliant. But the subtraction only works because every prompt carries exactly one answer from each model, so the mean is a perfectly balanced reference. In the hidden test each prompt carries a few passages, and the "mean" is just the other one or two.

Measured on the same OOF scores: centring with all 18 answers per prompt, versus centring within random draws of 2, 3 and 6 answers per prompt.

In [ ]:
S = OOF["char"]
C18 = bacc(y, CL[(S - pd.DataFrame(S).groupby(g).transform("mean").values).argmax(1)])
trap = []
for k in [2, 3, 6, 18]:
    plain, cent = [], []
    for r in range(20):
        i = train.groupby("prompt_id").sample(min(k, 17), random_state=r).index.values if k < 18 else np.arange(len(train))
        Si = S[i]; Sc = Si - pd.DataFrame(Si).groupby(g[i]).transform("mean").values
        plain.append(bacc(y[i], CL[Si.argmax(1)])); cent.append(bacc(y[i], CL[Sc.argmax(1)]))
        if k == 18: break
    trap.append({"answers per prompt": k, "plain": np.mean(plain), "prompt-centred": np.mean(cent)})
trap = pd.DataFrame(trap).set_index("answers per prompt"); C2 = trap.loc[2, "prompt-centred"]
fig, ax = plt.subplots(figsize=(11, 4.8))
trap.plot(marker="o", ax=ax, color=[C["ink"], C["red"]], lw=2)
for k, r in trap.iterrows():
    for col, c in [("plain", C["ink"]), ("prompt-centred", C["red"])]:
        ax.annotate(f"{r[col]:.3f}", (k, r[col]), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=9, color=c)
ax.set_xscale("log"); ax.set_xticks(trap.index); ax.set_xticklabels(trap.index); ax.set_ylabel("balanced accuracy")
ax.grid(color=C["grid"], lw=.6); ax.legend(frameon=False)
finish(fig, f"Centring adds +{C18-CHAR:.3f} with 18 answers per prompt and costs {C2-trap.loc[2,'plain']:+.3f} with 2",
       "Same out-of-fold character-model scores. 2, 3, 6: mean of 20 random draws per prompt. The hidden test has few passages per prompt.")
print(trap.round(4).to_string())

<a id='12'></a>
## 12 · What this means for modelling

**What the measurements confirm**
* The label is mostly in *style*, and the cheapest style signal (punctuation shape) already carries a large share of what character n-grams find.
* Lab is found far more often than model (67.5% vs 0.329). The leaderboard is decided **inside** families: five GPTs, four Geminis, five Claudes, two GLMs.
* Openings (section 06) are shared across models ("in the", "beneath the", "deep within"): the prompts ask for scene-like prose, so first words carry little author signal.
* Kimi-k3 mixes both habits (em-dash in 52% of passages, curly quotes in 37%); GLM's typography is indistinguishable from Claude's.
* Word features chase prompts. Anything trained on topic words learns the 150 training prompts, and the test has none of them.
* Prompt-group tricks need many answers per prompt. Validate them at 2 per prompt or not at all.

**What remains a hypothesis**
* That the typography habits hold at the same rates on the hidden test (same generation pipeline is likely, not verified: the interactive test file has 3 rows).
* That GLM's closeness to Claude (section 04, section 10) reflects shared training data rather than chance in 150 prompts.

**What this notebook does not claim**
* These baselines are not competitive (the public board is near 0.68 with large fine-tuned language models and external data). They are the honest floor, on folds that every later notebook will share.
* No adversarial validation against the test set: the hidden test is not available to notebooks outside the scoring rerun.

### Next in this series
Honest CPU baseline on these folds with external passages added one source at a time → a fine-tuned language-model classifier → stack, each scored on 2 answers per prompt.

### Results ledger (Team Taskers, public board)
| submission | what it is | public LB |
|---|---|---|
| v1 TF-IDF char 1-5 + word 1-2, SGD | sparse text, official data × 3 + open pool 212k | 0.422 |
| public pipeline (alexchilton 0.670) | run unchanged | 0.670 |
| public pipeline (lightningv08 0.680) | run unchanged | 0.680 |
| this notebook (character LR, official data only) | written as `submission.parquet`, not submitted | - |

If this helped you validate your own model, an upvote keeps the series going.

## Appendix · Submission file

A code competition needs `submission.parquet`. The character view is refitted on all 2,699 passages and predicts the test file (3 rows interactively, ~4,000 in the scoring rerun). It is a floor, not a contender.

In [ ]:
vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), sublinear_tf=True, min_df=2, max_features=300000)
clf = LogisticRegression(C=10, max_iter=3000).fit(vec.fit_transform(train.text), y)
sub = pd.DataFrame({"id": test["id"], "model": clf.predict(vec.transform(test.text.fillna("")))})
assert sub.id.is_unique and sub.model.isin(MODELS).all() and sub.notna().all().all()
sub.to_parquet("submission.parquet", index=False); print(sub.shape); print(sub.head())